In [141]:
import json
from typing import List, Tuple

In [143]:
txt_path = "ENACTING_Day2_Messi (1).txt"

json_path = "gpt4.json"

with open(txt_path, "r", encoding="utf-8") as f:
    data = json.load(f)  

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=4, ensure_ascii=False)

print("Successfully converted to:", json_path)

Successfully converted to: gpt4.json


In [145]:
# Utility: Convert HH:MM:SS timestamp into seconds
def time_to_seconds(timestamp: str) -> int:
    """
    Convert a timestamp in HH:MM:SS format into total seconds.

    Example:
        "00:06:28" -> 388 seconds
    """
    hours, minutes, seconds = map(int, timestamp.split(":"))
    return hours * 3600 + minutes * 60 + seconds

In [147]:
# Load segments from JSON file and convert to second intervals
def load_segments(path: str) -> List[Tuple[int, int]]:
    """
    Load segments from a JSON file and convert them into
    (start_time, end_time) tuples in seconds.

    Only segments with end_time > start_time are kept.
    """
    with open(path, "r") as f:
        data = json.load(f)

    segments = []
    for seg in data["segments"]:
        start = time_to_seconds(seg["time_in"])
        end = time_to_seconds(seg["time_out"])

        # Filter out invalid segments (e.g., reversed time boundaries)
        if end > start:
            segments.append((start, end))

    return segments

In [149]:
# Count number of segments
def segment_count(segments: List[Tuple[int, int]]) -> int:
    """
    Return total number of temporal segments.
    """
    return len(segments)

In [151]:
# Merge overlapping intervals
def merge_intervals(segments: List[Tuple[int, int]]) -> List[Tuple[int, int]]:
    """
    Merge overlapping time intervals into non-overlapping intervals.

    This is required before computing total temporal coverage.
    """
    if not segments:
        return []

    segments = sorted(segments)
    merged = [segments[0]]

    for current_start, current_end in segments[1:]:
        last_start, last_end = merged[-1]

        # If intervals overlap, merge them
        if current_start <= last_end:
            merged[-1] = (last_start, max(last_end, current_end))
        else:
            merged.append((current_start, current_end))

    return merged

In [153]:
# Compute total temporal coverage (after merging overlaps)
def total_coverage(segments: List[Tuple[int, int]]) -> int:
    """
    Compute total duration (in seconds) covered by segments.
    Overlapping intervals are merged before summation.
    """
    merged = merge_intervals(segments)
    return sum(end - start for start, end in merged)

In [155]:
# Compute total temporal intersection between two segment sets
def intersection_time(
    segments1: List[Tuple[int, int]],
    segments2: List[Tuple[int, int]]
) -> int:
    """
    Compute total overlapping duration (in seconds)
    between two sets of temporal segments.
    """
    total = 0

    for s1, e1 in segments1:
        for s2, e2 in segments2:
            overlap = max(0, min(e1, e2) - max(s1, s2))
            total += overlap

    return total


In [157]:
# Compute Intersection over Union 
def intersection_time(
    segments1: List[Tuple[int, int]],
    segments2: List[Tuple[int, int]]
) -> int:
    """
    Compute total intersection time between two segment sets.
    Uses merged segments to avoid double counting.
    """

    seg1 = merge_intervals(segments1)
    seg2 = merge_intervals(segments2)

    i, j = 0, 0
    intersection = 0

    while i < len(seg1) and j < len(seg2):
        start1, end1 = seg1[i]
        start2, end2 = seg2[j]

        # Compute overlap
        start = max(start1, start2)
        end = min(end1, end2)

        if start < end:
            intersection += end - start

        # Move pointer
        if end1 < end2:
            i += 1
        else:
            j += 1

    return intersection


In [159]:
# Compute average boundary deviation
def boundary_deviation(
    segments1: List[Tuple[int, int]],
    segments2: List[Tuple[int, int]]
) -> float:
    """
    Compute average absolute deviation (in seconds)
    between overlapping segment boundaries.

    For every overlapping pair, measure:
        |start1 - start2|
        |end1 - end2|
    """
    deviations = []

    for s1, e1 in segments1:
        for s2, e2 in segments2:
            overlap = min(e1, e2) - max(s1, s2)

            if overlap > 0:
                deviations.append(abs(s1 - s2))
                deviations.append(abs(e1 - e2))

    if not deviations:
        return 0.0

    return sum(deviations) / len(deviations)

In [161]:
#               RUN EVALUATION
# ============================================================

gpt4_segments = load_segments("gpt4.json")
rule_segments = load_segments("gpt5.2.json")


print("Segment Count Comparison")
print("GPT-4:", segment_count(gpt4_segments))
print("GPT-5.2:", segment_count(rule_segments))

print("Total Temporal Coverage (seconds)")
print("GPT-4:", total_coverage(gpt4_segments))
print("GPT-5.2:", total_coverage(rule_segments))

print("Temporal Intersection over Union (IoU)")
print("IoU:", compute_iou(gpt4_segments, rule_segments))

print("Average Boundary Deviation (seconds)")
print("Deviation:", boundary_deviation(gpt4_segments, rule_segments))

Segment Count Comparison
GPT-4: 43
GPT-5.2: 189
Total Temporal Coverage (seconds)
GPT-4: 316
GPT-5.2: 570
Temporal Intersection over Union (IoU)
IoU: 0.5543859649122806
Average Boundary Deviation (seconds)
Deviation: 49.741123683183766
